# Custom Python Calculation

Run your own Python script against one or more materials on the Mat3ra platform. The script and any
files it needs are uploaded to your object storage folder, a workflow fetches them onto the compute
node alongside the material, and whatever the script prints comes back as the result.

<h2 style="color:green">Usage</h2>

1. Put your script, its dependencies and any data files it reads in cell 1.2. below (or use the
   default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the
   script, its dependencies, its asset files, its settings, the materials, compute resources and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then
   select account and project.
1. Create materials: materials are read from the `../uploads` folder — place files there manually or
   run a material creation notebook first. If a material is not found by name, Standata is used as a
   fallback. Each material is then saved to the platform.
1. Create workflow: upload the script and its asset files, then assemble a workflow that fetches
   them, fetches the material, and runs the script. Optionally save the workflow to the collection.
1. Configure compute: get list of clusters and create compute configuration with selected cluster,
   queue, and number of processors.
1. Create one job per material from the material, workflow, project and compute configuration.
1. Submit the jobs and monitor the status: submit and wait for completion.
1. Retrieve results: read each job's standard output and display the values it printed.

## How the script receives its inputs

Everything lands in the job's working directory, so the script reads it all by **relative path**:

| File | Written by | Contents |
| --- | --- | --- |
| `material.json` | the workflow | the job's material, as stored on the platform |
| `settings.json` | this notebook | the `SETTINGS` dictionary below, uploaded next to the script |
| your asset files | this notebook | uploaded verbatim from `USER_ASSET_FILES` |

The script's standard output is the result. Print JSON and this notebook renders it as a table.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and job

`USER_ASSET_FILES` names files your script opens. Put them in the `../uploads` folder first — drag
them into the JupyterLite file browser — and this notebook uploads them alongside your script.

In [ ]:
import json
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAMES = ["Silicon"]  # One job is created per material

# 4. Script parameters
USER_REQUIREMENTS = ["numpy<2"]  # Installed into a virtual environment on the compute node
USER_ASSET_FILES = ["radii.json"]  # Files the script opens, taken from FOLDER
SETTINGS = {"cutoff_scale": 1.2}  # Uploaded as settings.json next to the script

# 5. Workflow parameters
WORKFLOW_SEARCH_TERM = "custom_script.json"  # Search term for Workflows Standata
APPLICATION_NAME = "python"
MY_WORKFLOW_NAME = "Custom Python Calculation"
save_to_collection = True

# 6. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 7. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set the script to run

This is the calculation. It runs on the compute node with `material.json`, `settings.json` and your
asset files beside it, and whatever it prints becomes the result. Replace it with your own.

The script is uploaded as a file and fetched onto the node, never inlined into the workflow, so its
contents reach Python exactly as written - text that looks like a template placeholder is left
alone.

In [ ]:
USER_SCRIPT = r"""
import itertools
import json

import numpy as np

material = json.load(open("material.json"))
settings = json.load(open("settings.json"))
radii = json.load(open("radii.json"))

# The platform stores the cell as lengths and angles, so build the vectors from them.
lattice = material["lattice"]
a, b, c = lattice["a"], lattice["b"], lattice["c"]
alpha, beta, gamma = (np.radians(lattice[key]) for key in ("alpha", "beta", "gamma"))
c_x = c * np.cos(beta)
c_y = c * (np.cos(alpha) - np.cos(beta) * np.cos(gamma)) / np.sin(gamma)
vectors = np.array(
    [
        [a, 0.0, 0.0],
        [b * np.cos(gamma), b * np.sin(gamma), 0.0],
        [c_x, c_y, np.sqrt(max(c**2 - c_x**2 - c_y**2, 0.0))],
    ]
)

elements = [element["value"] for element in material["basis"]["elements"]]
crystal = np.array([point["value"] for point in material["basis"]["coordinates"]], dtype=float)
cartesian = crystal @ vectors

# Count neighbours within scale * (r_i + r_j), including atoms in the neighbouring cells.
scale = settings["cutoff_scale"]
images = [np.array(shift) @ vectors for shift in itertools.product((-1, 0, 1), repeat=3)]

coordination = {}
for element_i, position_i in zip(elements, cartesian):
    neighbors = 0
    for element_j, position_j in zip(elements, cartesian):
        cutoff = scale * (radii[element_i] + radii[element_j])
        for image in images:
            distance = np.linalg.norm(position_i - position_j - image)
            if 0.01 < distance < cutoff:
                neighbors += 1
    coordination[element_i] = neighbors

print(json.dumps({
    "formula": material.get("formula"),
    "n_atoms": len(elements),
    "coordination": coordination,
}))
"""

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Using account: {selected_account.name} ({ACCOUNT_ID})")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Load materials from local files (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

materials = [
    load_material_from_folder(FOLDER, name) or Material.create(Materials.get_by_name_first_match(name))
    for name in MATERIAL_NAMES
]
visualize([{"material": material, "title": material.name} for material in materials])

### 3.2. Save materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [
    Material.create(get_or_create_material(client, material, ACCOUNT_ID)) for material in materials
]

## 4. Create workflow and set its parameters
### 4.1. Upload the script and its asset files

The files go to your account's object storage folder ("Dropbox"), which the compute node reads them
from. An upload travels inside the request body, which the API caps at 50 MB, and JupyterLite holds
the content in memory before sending it. For anything large, upload it through the Dropbox page in
the web interface instead and name it in `USER_ASSET_FILES` all the same.

In [ ]:
import os

from mat3ra.notebooks_utils.core.entity.file.api import upload_files

files_to_upload = {"user_script.py": USER_SCRIPT, "settings.json": json.dumps(SETTINGS)}
for name in USER_ASSET_FILES:
    if name in files_to_upload:
        raise ValueError(f"Rename '{name}' in USER_ASSET_FILES: this notebook already uploads a file by that name.")
    path = os.path.join(FOLDER, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"'{name}' is listed in USER_ASSET_FILES but is not in {FOLDER}.")
    with open(path) as file:
        files_to_upload[name] = file.read()

uploaded_files = upload_files(client, files_to_upload, ACCOUNT_ID)

### 4.2. Create workflow from standard workflows and preview it

The `Custom Python Script` workflow already carries the unit chain this needs: fetch the uploaded
files, fetch the material, run the script. Three things are filled in per job — the objects to
fetch, the runner that hands your script its inputs, and the dependency list.

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.core.entity.file.api import to_object_storage_input
from mat3ra.notebooks_utils.core.entity.workflow.api import CUSTOM_SCRIPT_RUNNER, set_execution_unit_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow_config["name"] = MY_WORKFLOW_NAME
subworkflow = workflow_config["subworkflows"][0]
subworkflow["name"] = MY_WORKFLOW_NAME
units = {unit["name"]: unit for unit in subworkflow["units"]}

units["io-user-files"]["input"] = [to_object_storage_input(file) for file in uploaded_files]

set_execution_unit_input(units["custom_script"], "script.py", CUSTOM_SCRIPT_RUNNER)
set_execution_unit_input(units["custom_script"], "requirements.txt", "\n".join(USER_REQUIREMENTS) + "\n")

workflow = Workflow.create(workflow_config)
visualize_workflow(workflow)

### 4.3. Save workflow to collection

Saving it makes the workflow reusable: it stays in your collection pointing at the uploaded files,
so it can be run again — from the UI or another notebook — against any material. To change the
script's parameters, upload a new `settings.json` over the old one.

In [ ]:
saved_workflow = None
if save_to_collection:
    saved_workflow = Workflow.create(
        client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
    )
    print(f"✅ Workflow saved to collection: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN,
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the jobs with material and workflow configuration
### 6.1. Create one job per material

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

jobs = []
for saved_material in saved_materials:
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=saved_workflow or workflow,
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}",
        compute=compute.to_dict(),
    )
    jobs.append(job_response if not isinstance(job_response, list) else job_response[0])

job_ids = [job["_id"] for job in jobs]
print(f"✅ Created {len(job_ids)} jobs: {job_ids}")
display_JSON(jobs[0])

## 7. Submit the jobs and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, job_ids)
print(f"✅ Submitted {len(job_ids)} jobs successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 8. Retrieve results

Each job's standard output is the script's own output. Lines that parse as JSON become table
columns; everything else is shown as printed.

In [ ]:
import pandas as pd

from mat3ra.notebooks_utils.io import read_from_url


async def read_job_stdout(job_id):
    """Contents of the execution unit's .out file, or why the job produced none."""
    files = client.jobs.list_files(job_id)
    stdout_file = next((file for file in files if file["key"].endswith(".out")), None)
    if stdout_file is None:
        job = client.jobs.get(job_id)
        errors = job.get("compute", {}).get("errors", [])
        return f"No output. Job status: {job['status']}. {json.dumps(errors, indent=2)}"
    return await read_from_url(stdout_file["signedUrl"])


results = []
for saved_material, job in zip(saved_materials, jobs):
    stdout = await read_job_stdout(job["_id"])
    print(f"--- {job['name']} ---\n{stdout}")
    for line in stdout.splitlines():
        try:
            results.append({"material": saved_material.name, **json.loads(line)})
        except json.JSONDecodeError:
            continue

pd.DataFrame(results)